In [2]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


raw_file_path = Path("../Data/Raw/retail_sales_transactions.csv")

clean_df = pd.read_csv(raw_file_path,
    dtype={
        "Invoice": "string",
        "StockCode": "string"
    },
    parse_dates=["InvoiceDate"])

# Restore nullable integer datatype after loading CSV
clean_df["CustomerID"] = pd.to_numeric(clean_df["CustomerID"],errors="coerce").astype("Int64")

print("Shape:", clean_df.shape)

Shape: (1028761, 13)


In [3]:
# ---------------------------------------------------------
# DAY 2 — TASK 1: BASELINE DATA AUDIT
# ---------------------------------------------------------

print("📊 DATASET OVERVIEW")
print("----------------------------")
print("Shape:", clean_df.shape)

print("\nColumns:")
print(clean_df.columns.tolist())

# MISSING VALUES REPORT

missing_report = pd.DataFrame({
    "MissingValues": clean_df.isna().sum(),
    "MissingPercentage": (clean_df.isna().mean() * 100).round(2)
})


# TRANSACTION QUALITY CHECKS

guest_rows = clean_df["CustomerID"].isna().sum()

cancelled_rows = (clean_df["Invoice"].astype("string").str.upper().str.startswith("C", na=False).sum())

return_rows = (clean_df["Quantity"] < 0).sum()
zero_quantity_rows = (clean_df["Quantity"] == 0).sum()
invalid_price_rows = (clean_df["UnitPrice"] <= 0).sum()
missing_description_rows = (clean_df["Description"].isna()).sum()
invalid_date_rows = (clean_df["InvoiceDate"].isna()).sum()

print("\nCLEANING AUDIT")
print("----------------------------")
print(f"Guest checkout rows:        {guest_rows:,}")
print(f"Cancelled rows:             {cancelled_rows:,}")
print(f"Return rows:                {return_rows:,}")
print(f"Zero quantity rows:         {zero_quantity_rows:,}")
print(f"Non-positive price rows:    {invalid_price_rows:,}")
print(f"Missing description rows:   {missing_description_rows:,}")
print(f"Invalid date rows:          {invalid_date_rows:,}")

📊 DATASET OVERVIEW
----------------------------
Shape: (1028761, 13)

Columns:
['Invoice', 'InvoiceDate', 'CustomerID', 'StockCode', 'Description', 'Quantity', 'UnitPrice', 'Country', 'IsGuestCheckout', 'IsCancelled', 'IsReturn', 'TransactionType', 'TotalPrice']

CLEANING AUDIT
----------------------------
Guest checkout rows:        230,876
Cancelled rows:             19,104
Return rows:                19,863
Zero quantity rows:         0
Non-positive price rows:    1,744
Missing description rows:   0
Invalid date rows:          0


In [4]:
clean_df.dropna(subset=["CustomerID"], inplace=True)

In [5]:
cancelled_mask = (
    clean_df["Invoice"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.startswith("C", na=False)
)

clean_df.drop(
    index=clean_df[cancelled_mask].index,
    inplace=True
)

In [6]:
clean_df.drop(index=clean_df[clean_df["UnitPrice"] <= 0].index,inplace=True)

In [7]:
guest_rows = clean_df["CustomerID"].isna().sum()

cancelled_rows = (clean_df["Invoice"].astype("string").str.upper().str.startswith("C", na=False).sum())

return_rows = (clean_df["Quantity"] < 0).sum()
zero_quantity_rows = (clean_df["Quantity"] == 0).sum()
invalid_price_rows = (clean_df["UnitPrice"] <= 0).sum()
missing_description_rows = (clean_df["Description"].isna()).sum()
invalid_date_rows = (clean_df["InvoiceDate"].isna()).sum()

print("\nCLEANING AUDIT")
print("----------------------------")
print(f"Guest checkout rows:        {guest_rows:,}")
print(f"Cancelled rows:             {cancelled_rows:,}")
print(f"Return rows:                {return_rows:,}")
print(f"Zero quantity rows:         {zero_quantity_rows:,}")
print(f"Non-positive price rows:    {invalid_price_rows:,}")
print(f"Missing description rows:   {missing_description_rows:,}")
print(f"Invalid date rows:          {invalid_date_rows:,}")


CLEANING AUDIT
----------------------------
Guest checkout rows:        0
Cancelled rows:             0
Return rows:                0
Zero quantity rows:         0
Non-positive price rows:    0
Missing description rows:   0
Invalid date rows:          0


Here I drop IsGuestCheckout Column But I delete that cell by mistake,and after deletion we not able to ReRun that cell,so that's why it's not mentioned.

In [8]:
clean_df.drop(columns=["IsCancelled","IsReturn","TransactionType"],inplace = True)

In [9]:
from pathlib import Path

processed_path = Path("../Data/Processed/retail_sales_cleaned.csv")

clean_df.to_csv(processed_path,index=False)

print("✅ Saved:", processed_path)

✅ Saved: ..\Data\Processed\retail_sales_cleaned.csv


# Feature Engineering: Customer RFM Analysis
Core RFM metrics:
- Recency   : Days since the customer's latest purchase
- Frequency : Number of unique invoices placed by the customer
- Monetary  : Total revenue generated by the customer

In [11]:
# Load The Data
clean_df = pd.read_csv(
    processed_path,
    dtype={"Invoice": "string", "StockCode": "string"},
    parse_dates=["InvoiceDate"]
)

# Restore nullable integer datatype after loading CSV
clean_df["CustomerID"] = pd.to_numeric(clean_df["CustomerID"],errors="coerce").astype("Int64")

print("Shape:", clean_df.shape)
print("\nColumns:",clean_df.columns.tolist())
print("\nData types:")
print(clean_df.dtypes)

Shape: (779425, 10)

Columns: ['Invoice', 'InvoiceDate', 'CustomerID', 'StockCode', 'Description', 'Quantity', 'UnitPrice', 'Country', 'IsGuestCheckout', 'TotalPrice']

Data types:
Invoice                    string
InvoiceDate        datetime64[us]
CustomerID                  Int64
StockCode                  string
Description                   str
Quantity                    int64
UnitPrice                 float64
Country                       str
IsGuestCheckout              bool
TotalPrice                float64
dtype: object


In [12]:
print("Total transaction rows:", len(clean_df))
print("Unique customers:", clean_df["CustomerID"].nunique())
print("Missing Customer IDs:", clean_df["CustomerID"].isna().sum())
print("Unique invoices:", clean_df["Invoice"].nunique())

# Use one day after the final transaction date
analysis_date = (clean_df["InvoiceDate"].max().normalize()+ pd.Timedelta(days=1))
print("\nLatest transaction date:", clean_df["InvoiceDate"].max())
print("RFM analysis date:", analysis_date)

Total transaction rows: 779425
Unique customers: 5878
Missing Customer IDs: 0
Unique invoices: 36969

Latest transaction date: 2011-12-09 12:50:00
RFM analysis date: 2011-12-10 00:00:00


In [13]:
def get_primary_country(country_series):
    country_series = country_series.dropna()

    if country_series.empty:
        return pd.NA

    return country_series.mode().iloc[0]

In [14]:
rfm_df = (clean_df.groupby("CustomerID", as_index=False).agg(
        PrimaryCountry=("Country", get_primary_country),
        FirstPurchaseDate=("InvoiceDate", "min"),
        LastPurchaseDate=("InvoiceDate", "max"),
        Frequency=("Invoice", "nunique"),
        Monetary=("TotalPrice", "sum"),
        TotalQuantity=("Quantity", "sum"))
)

# Recency: lower value means a more recently active customer
rfm_df["Recency"] = (analysis_date - rfm_df["LastPurchaseDate"].dt.normalize()).dt.days

# Average amount spent per unique order
rfm_df["AverageOrderValue"] = (rfm_df["Monetary"] / rfm_df["Frequency"]).round(2)

# Number of days between first and latest purchases
rfm_df["CustomerLifetimeDays"] = (rfm_df["LastPurchaseDate"].dt.normalize()
    - rfm_df["FirstPurchaseDate"].dt.normalize()
    ).dt.days

# Arrange the columns in the required order
rfm_df = rfm_df[["CustomerID","PrimaryCountry","FirstPurchaseDate","LastPurchaseDate","Recency","Frequency","Monetary",
    "TotalQuantity","AverageOrderValue","CustomerLifetimeDays"]]

print("Base RFM dataset created successfully.")
print("Shape:", rfm_df.shape)

rfm_df.head()

Base RFM dataset created successfully.
Shape: (5878, 10)


,CustomerID,PrimaryCountry,FirstPurchaseDate,LastPurchaseDate,Recency,Frequency,Monetary,TotalQuantity,AverageOrderValue,CustomerLifetimeDays
0,12346,United Kingdom,2009-12-14 08:34:00,2011-01-18 10:01:00,326,12,77556.46,74285,6463.04,400
1,12347,Iceland,2010-10-31 14:20:00,2011-12-07 15:52:00,3,8,4921.53,2967,615.19,402
2,12348,Finland,2010-09-27 14:59:00,2011-09-25 13:13:00,76,5,2019.40,2714,403.88,363
3,12349,Italy,2010-04-29 13:20:00,2011-11-21 09:51:00,19,4,4428.69,1624,1107.17,571
4,12350,Norway,2011-02-02 16:01:00,2011-02-02 16:01:00,311,1,334.40,197,334.40,0


In [15]:
# RFM SCORE FEATURE ENGINEERING

# Recency:
# Lower recency is better, so the lowest values receive score 5.
rfm_df["RecencyScore"] = pd.qcut(rfm_df["Recency"].rank(method="first",ascending=True)
                                 ,q=5,labels=[5, 4, 3, 2, 1]).astype("int64")

# Frequency:
# Higher number of unique orders is better.
rfm_df["FrequencyScore"] = pd.qcut(rfm_df["Frequency"].rank(method="first",ascending=True)
                                   ,q=5,labels=[1, 2, 3, 4, 5]).astype("int64")

# Monetary:
# Higher total customer spending is better.
rfm_df["MonetaryScore"] = pd.qcut(rfm_df["Monetary"].rank(method="first",ascending=True)
                                  ,q=5,labels=[1, 2, 3, 4, 5]).astype("int64")


# Combine the three scores as text.
# Example: RecencyScore=5, FrequencyScore=4, MonetaryScore=3
# becomes RFMScore="543".
rfm_df["RFMScore"] = (rfm_df["RecencyScore"].astype(str)
    + rfm_df["FrequencyScore"].astype(str)
    + rfm_df["MonetaryScore"].astype(str))

In [16]:
processed_rfm_path = Path("../Data/Processed/retail_sales_rfm.csv")

rfm_df.to_csv(processed_rfm_path,index=False)
print("Saved file:", processed_rfm_path)
print("Shape:", rfm_df.shape)

Saved file: ..\Data\Processed\retail_sales_rfm.csv
Shape: (5878, 14)


In [17]:
rfm_df.head(10)

,CustomerID,PrimaryCountry,FirstPurchaseDate,LastPurchaseDate,Recency,Frequency,Monetary,TotalQuantity,AverageOrderValue,CustomerLifetimeDays,RecencyScore,FrequencyScore,MonetaryScore,RFMScore
0,12346,United Kingdom,2009-12-14 08:34:00,2011-01-18 10:01:00,326,12,77556.46,74285,6463.04,400,2,5,5,255
1,12347,Iceland,2010-10-31 14:20:00,2011-12-07 15:52:00,3,8,4921.53,2967,615.19,402,5,4,5,545
2,12348,Finland,2010-09-27 14:59:00,2011-09-25 13:13:00,76,5,2019.40,2714,403.88,363,3,4,4,344
3,12349,Italy,2010-04-29 13:20:00,2011-11-21 09:51:00,19,4,4428.69,1624,1107.17,571,5,3,5,535
4,12350,Norway,2011-02-02 16:01:00,2011-02-02 16:01:00,311,1,334.40,197,334.40,0,2,1,2,212
5,12351,Unspecified,2010-11-29 15:23:00,2010-11-29 15:23:00,376,1,300.93,261,300.93,0,2,1,2,212
6,12352,Norway,2010-11-12 10:20:00,2011-11-03 14:37:00,37,10,2849.84,724,284.98,356,4,5,4,454
7,12353,Bahrain,2010-10-27 12:44:00,2011-05-19 17:47:00,205,2,406.76,212,203.38,204,2,2,2,222
8,12354,Spain,2011-04-21 13:11:00,2011-04-21 13:11:00,233,1,1079.40,530,1079.40,0,2,1,3,213
9,12355,Bahrain,2010-05-21 11:59:00,2011-05-09 13:49:00,215,2,947.61,543,473.80,353,2,2,3,223
